In [1]:
import os
import random
import numpy as np 
import pandas as pd

import cv2
from PIL import Image
import albumentations as A

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import torch
from tqdm.notebook import tqdm
from torch.utils.data import Dataset
from torchvision import transforms as T
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
set_seed(0)

In [3]:
def create_df(IMAGE_PATH):
    name = []
    for dirname, _, filenames in os.walk(IMAGE_PATH):
        for filename in filenames:
            name.append(filename.split('.')[0])
    return pd.DataFrame({'id': name}, index = np.arange(0, len(name)))

In [4]:
batch_size = 4
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

In [5]:
class CloudTestDataset(Dataset):
    
    def __init__(self, img_path, X, transform=None):
        self.img_path = img_path
        self.X = X
        self.transform = transform
      
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        img = cv2.imread(self.img_path + self.X[idx] + '.jpg')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_name = self.X[idx]

        if self.transform is not None:
            aug = self.transform(image=img)
            img = Image.fromarray(aug['image'])
        
        if self.transform is None:
            img = Image.fromarray(img)
        
        return img, img_name

In [6]:
model = smp.Unet('timm-mobilenetv3_large_100', encoder_weights='imagenet', classes=8,
                 activation=None, encoder_depth=5, decoder_channels=[256, 128, 64, 32, 16])
model.load_state_dict(torch.load('./Unet-wiou-0.721_loss-0.777.pt', weights_only=False))

<All keys matched successfully>

### Generate segmented masks for each cluster

In [7]:
discrete_cmap = ListedColormap([
    [  0/255,   0/255,   0/255], # Clear: Black
    [255/255,  75/255,  75/255], # Cirrus: Light Red
    [255/255, 150/255,   0/255], # Cirrostratus: Light Orange
    [255/255, 210/255,  75/255], # Stratus: Light Yellow
    [175/255, 225/255, 130/255], # Stratocumulus: Light Green
    [130/255, 175/255, 225/255], # Cumulus: Light Blue
    [175/255, 150/255, 255/255], # Cirrocumulus: Light Purple
    [150/255, 150/255, 150/255]  # Nimbus: Gray
])

In [ ]:
for c in range(1, 4):
    cluster = c

    IMAGE_PATH_TEST = f'./clustered_images/cluster_{cluster}/'
    output_path = f'./clustered_masks/cluster_{cluster}_masks/'
    os.makedirs(output_path, exist_ok=True)

    df_test = create_df(IMAGE_PATH_TEST)
    X_test = df_test['id'].values
    t_test = A.Resize(320, 416, interpolation=cv2.INTER_AREA)
    test_set = CloudTestDataset(IMAGE_PATH_TEST, X_test, transform=t_test)

    for i in tqdm(range(len(test_set))):
        image, image_name = test_set[i]

        model.eval()
        t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
        image = t(image)
        model.to(device)
        image = image.to(device)

        with torch.no_grad():
            image = image.unsqueeze(0)
            output = model(image)
            masked = torch.argmax(output, dim=1)
            masked = masked.cpu().squeeze(0)

        save_name = os.path.join(output_path, f"{image_name}_mask.jpg")
        plt.imsave(save_name, masked.numpy(), cmap=discrete_cmap, vmin=0, vmax=7)

### Generate HSV mean values for each cluster

In [ ]:
# all_results = []
# for num in range(1, 4):
#     cluster = num
#     IMAGE_PATH_TEST = f'./clustered_images/cluster_{cluster}/'
#     df_test = create_df(IMAGE_PATH_TEST)
#     X_test = df_test['id'].values
#     t_test = A.Resize(320, 416, interpolation=cv2.INTER_AREA)
#     test_set = CloudTestDataset(IMAGE_PATH_TEST, X_test, transform=t_test)

#     for i in tqdm(range(len(test_set))):
#         image, image_name = test_set[i]
#         rgb_image = np.array(image.convert("RGB"))
#         hsv_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2HSV)

#         h_mean = np.mean(hsv_image[:, :, 0])
#         s_mean = np.mean(hsv_image[:, :, 1])
#         v_mean = np.mean(hsv_image[:, :, 2])

#         result_row = {
#             "img_name": image_name,
#             "h_mean": round(h_mean, 3),
#             "s_mean": round(s_mean, 3),
#             "v_mean": round(v_mean, 3)
#         }
#         all_results.append(result_row)

# save_dir = "../Relationship_Study_Between_Cloud_and_Radiation"
# os.makedirs(save_dir, exist_ok=True)
# df_all = pd.DataFrame(all_results)
# df_all.to_csv(os.path.join(save_dir, "hsv_mean.csv"), index=False)

### Calculate cloud attributes

In [ ]:
all_results = []
for num in range(1, 4):
    cluster = num
    IMAGE_PATH_TEST = f'./clustered_images/cluster_{cluster}/'
    df_test = create_df(IMAGE_PATH_TEST)
    X_test = df_test['id'].values
    t_test = A.Resize(320, 416, interpolation=cv2.INTER_AREA)
    test_set = CloudTestDataset(IMAGE_PATH_TEST, X_test, transform=t_test)

    label_names = [
        "Clear", "Cirrus", "Cirrostratus", "Stratus",
        "Stratocumulus", "Cumulus", "Cirrocumulus", "Nimbus"
    ]

    for i in tqdm(range(len(test_set))):
        image, image_name = test_set[i]
        model.eval()
        t = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
        image_tensor = t(image).unsqueeze(0).to(device)
        model.to(device)

        with torch.no_grad():
            output = model(image_tensor)  # shape: (1, C, H, W)
            masked = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

        # Label ratio
        total_pixels = masked.size
        label_counts = np.bincount(masked.flatten(), minlength=8)
        label_ratios = label_counts / total_pixels

        result_row = {"image_name": image_name}
        for label in range(8):
            ratio = round(label_ratios[label], 3)
            result_row[label_names[label]] = ratio

        # get V channel
        rgb_image = np.array(image.convert("RGB"))
        hsv_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2HSV)
        v_channel = hsv_image[:, :, 2] / 255.0  # Normalize to [0, 1]
        # weighted sum
        weighted_sums = np.zeros(8)
        for i in range(masked.shape[0]):
            for j in range(masked.shape[1]):
                label_idx = masked[i, j]
                v = v_channel[i, j]
                if label_idx == 0:  # Clear
                    weighted_sums[label_idx] += (1 - v)
                else:
                    weighted_sums[label_idx] += v

        for label in range(8):
            weighted = round(weighted_sums[label], 3)
            result_row[f"{label_names[label]}_weighted"] = weighted / total_pixels
        
        result_row["label"] = cluster

        all_results.append(result_row)

save_dir = "../Relationship_Study_Between_Cloud_and_Radiation"
os.makedirs(save_dir, exist_ok=True)
df_all = pd.DataFrame(all_results)
df_all.to_csv(os.path.join(save_dir, "img_attribute.csv"), index=False)